<a href="https://colab.research.google.com/github/samanthafaith00001-png/Personal-Finance-Analysis/blob/main/Advanced_Financial_Intelligence_dashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [23]:
from numpy.random import randint
import pandas as pd
import numpy as np

np.random.seed(42)

categories = []

for i in range(500):

    categories = [
        "Salary", "Business", "Food",
        " Transport", "Rent", "Shopping ",
        "Utilities", "Healthcare",
        "Education", "Entertainment"
    ]

    transactions = []

    for i in range(500):

        category = np.random.choice(categories)

        transaction_type = (
            "income" if category in ["Salary", "Business"]
            else "expense"
        )

        amount = (
            np.random.randint(15000, 120000)
            if transaction_type == "income"
            else np.random.randint(300, 10000)

        )

        transactions.append({
            "id": i + 1,
            "category": category,
            "type": transaction_type,
            "amount": amount,
            "date": pd.Timestamp.today()
                - pd.Timedelta(days=np.random.randint(0, 365))
        })

    df = pd.DataFrame(transactions)

    df["date"] = df["date"].dt.strftime("%Y-%m-%d")

    df.to_csv("transactions.csv", index=False)

    print("500 transaction generated successfully!")
    print(df.head(0))
    print(df.describe())




Streaming output truncated to the last 5000 lines.
count  500.000000     500.000000
mean   250.500000   19039.342000
std    144.481833   30289.246857
min      1.000000     323.000000
25%    125.750000    3587.000000
50%    250.500000    6605.000000
75%    375.250000    9397.000000
max    500.000000  119768.000000
500 transaction generated successfully!
Empty DataFrame
Columns: [id, category, type, amount, date]
Index: []
               id       amount
count  500.000000     500.0000
mean   250.500000   16766.9980
std    144.481833   27202.2495
min      1.000000     442.0000
25%    125.750000    3460.2500
50%    250.500000    6247.0000
75%    375.250000    9319.5000
max    500.000000  119692.0000
500 transaction generated successfully!
Empty DataFrame
Columns: [id, category, type, amount, date]
Index: []
               id         amount
count  500.000000     500.000000
mean   250.500000   18989.976000
std    144.481833   30605.549156
min      1.000000     308.000000
25%    125.750000    

In [28]:
import pandas as pd
import numpy as np

def analyze_finances(data) :

    df = pd.DataFrame(data)

    income = df.loc[
        df["type"] == "income", "amount"
    ].sum()

    expenses = df.loc[
        df["type"] =="expenses", "amount"
    ].sum()

    profit = income - expenses

    savings_rate = (
        (profit / income) * 100
        if income > 0 else 0
    )

    expenses_analysis = (
        df[df["type"] == "expense"]
        .groupby("category")["amount"]
        .sum()
        .sort_values(ascending=False)
    )

    return {
        "total_income": float(income),
        "total_expenses": float(expenses),
        "net_profit": float(profit),
        "savings_rate": round(float(savings_rate), 2),
        "highest_expense_category": (
            str(expense_analysis.index[0])
            if not expense_analysis.empty else "None"
        ),
        "average_transaction": float(
        np.mean(df["amount"])

        )
    }


In [38]:
!pip install flask flask-cors pandas numpy

from flask import Flask, jsonify, request
from flask_cors import CORS
import pandas as pd

app = Flask(__name__)
CORS(app)

DATA_FILE = "transaction.csv"

def load_data():
    df = pd.read_csv(DATA_FILE)
    df["date"] = df["date"].astype(str)
    return df


@app.route("/")
def home():
    return jsonify({
        "message": "Finacial Intelligence API",
        "status": "Running"
    })

@app.route("/api/transactions")
def get_transactions():
    df = load_data()
    return jsonify(df.to_dict(orient="records"))

@app.route("/api/transactions")
def get_analytics():
    df = load_data()

    results = analyze_finances(
      df.to_dict(orient="records")
    )

    return jsonify(results)

from threading import Thread

server_thread = Thread(
    target=lambda: app.run(
        host= "0.0.0.0",
        port=5000,
        debug=False,
        use_reloader=False
    )
)
server_thread.start()

print("Financial Intelligence API is STARTING...")

Financial Intelligence API is STARTING...
 * Serving Flask app '__main__'
 * Debug mode: off


In [40]:
from google.colab.output import eval_js

API_URL = eval_js(
    "google.colab.kernel.proxyPort(5000)"
)

print("Your API address:", API_URL)

Your API address: https://5000-m-s-usc1c0-25qwces2dcuq4-c.us-central1.prod.r.colab.dev


In [43]:
%%javascript

const API_URL = "http//127.0.0.1:5000";

async function loadFinancialAnalytics() {
    try {

        const response = await fetch(
            '${API_URL}/api/analytic'
        );

        if (!response.ok) {
            throw new Error("Could not load analytics");
        }

        const data = await response.json();

        document.getElementById("revenue").textContent =
            "KES " + data.total_income.toLocaleString();

        document.getElementById("expenses").textContent =
            "KES " + data.total_expense.toLocaleString();

        document.getElementById("profit").textContent =
            "KES " + data.net_profit.toLocaleString()

        document.getElementById("savings").textContent =
             data.savings_rate + "%";

        console.log("Finacial analytics:", data);

    } catch (error) {

        console.error("API ERROR:", error);
    }

}

loadFinancialAnalytics();

<IPython.core.display.Javascript object>